In [ ]:
!pip install -q transformers datasets faiss-gpu-cu12 accelerate bitsandbytes sentence-transformers
!pip install -q xformers peft nltk razdel beautifulsoup4 pandas tqdm


import os
import re
import gc
import csv
import pickle
import warnings
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


from bs4 import BeautifulSoup
from razdel import sentenize


import faiss
from sentence_transformers import SentenceTransformer


from transformers import AutoModelForSequenceClassification, AutoTokenizer


import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


SAMPLE_PATH = "/kaggle/input/datasets/mauron67/alfa-mfti-data/sample_submission.csv"
sample_sub = pd.read_csv(SAMPLE_PATH, quoting=csv.QUOTE_ALL)
print("Sample submission shape:", sample_sub.shape)
print("Columns:", sample_sub.columns.tolist())
print("Unique q_ids:", sample_sub['q_id'].nunique())
assert sample_sub.shape[0] == sample_sub['q_id'].nunique(), "Duplicate q_id found!"
print("✔ Each q_id appears exactly once.")


WEBSITES_PATH = "/kaggle/input/datasets/mauron67/alfa-mfti-data/websites.csv"
QUESTIONS_PATH = "/kaggle/input/datasets/mauron67/alfa-mfti-data/questions.csv"

websites = pd.read_csv(WEBSITES_PATH)
questions = pd.read_csv(QUESTIONS_PATH)

print(f"Websites: {websites.shape}")
print(f"Questions: {questions.shape}")


qid2query = dict(zip(questions['q_id'], questions['query']))


def clean_html(text: str) -> str:
    try:
        soup = BeautifulSoup(text, 'lxml')
        for element in soup(["script", "style", "nav", "footer", "header", "aside", "noscript"]):
            element.decompose()
        cleaned = soup.get_text(separator=' ')
    except Exception:
        cleaned = re.sub(r'<[^>]+>', ' ', text)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

def remove_boilerplate(text: str) -> str:
    text = re.sub(r'(Меню|Главная|Услуги|О банке|Контакты|Вакансии|Партнёрам).*?(?=\s\s|\n|$)', '', text)
    text = re.sub(r'©\s*\d{4}.*', '', text)
    text = re.sub(r'Все права защищены.*', '', text)
    text = re.sub(r'Телефон:\s*[\d\s\-\(\)\+]+', '', text)
    text = re.sub(r'^\W+$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n\s*\n', '\n', text)
    return text.strip()

def preprocess_text(raw: str) -> str:
    if not isinstance(raw, str):
        return ""
    return remove_boilerplate(clean_html(raw))

tqdm.pandas(desc="Cleaning")
websites['clean_text'] = websites['text'].progress_apply(preprocess_text)
websites = websites[websites['clean_text'].str.len() > 50].reset_index(drop=True)
print(f"Websites after cleaning: {len(websites)}")


def sentence_chunking(text: str, chunk_size: int = 3, overlap: int = 1) -> List[str]:
    sentences = [s.text for s in sentenize(text)]
    chunks = []
    if not sentences:
        return []
    for i in range(0, len(sentences), chunk_size - overlap):
        chunk = sentences[i:i + chunk_size]
        if chunk:
            chunks.append(' '.join(chunk))
        if i + chunk_size >= len(sentences):
            break
    return chunks

all_chunks = []
chunk_id = 0
for _, row in tqdm(websites.iterrows(), total=len(websites), desc="Chunking"):
    for ch in sentence_chunking(row['clean_text'], chunk_size=3, overlap=1):
        all_chunks.append({
            'chunk_id': chunk_id,
            'web_id': row['web_id'],
            'url': row.get('url', ''),
            'title': row.get('title', ''),
            'text': ch
        })
        chunk_id += 1

chunk_df = pd.DataFrame(all_chunks)
print(f"Total chunks: {len(chunk_df)}")


EMBED_MODEL_NAME = "intfloat/multilingual-e5-small"
INDEX_PATH = "/content/faiss_index.bin"
CHUNKS_PATH = "/content/chunks.pkl"

if os.path.exists(INDEX_PATH) and os.path.exists(CHUNKS_PATH):
    print("Loading index and chunks from disk...")
    index = faiss.read_index(INDEX_PATH)
    with open(CHUNKS_PATH, 'rb') as f:
        chunk_df = pickle.load(f)
    embedder = SentenceTransformer(EMBED_MODEL_NAME, device=device)
else:
    print("Computing embeddings...")
    embedder = SentenceTransformer(EMBED_MODEL_NAME, device=device)
    chunk_texts = ["passage: " + t for t in chunk_df['text'].tolist()]
    embeddings = embedder.encode(
        chunk_texts,
        batch_size=128,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    faiss.write_index(index, INDEX_PATH)
    with open(CHUNKS_PATH, 'wb') as f:
        pickle.dump(chunk_df, f)
    del embeddings, chunk_texts
    gc.collect()
    torch.cuda.empty_cache()

print(f"Index contains {index.ntotal} vectors")


RERANKER_NAME = "BAAI/bge-reranker-v2-m3"
reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_NAME)
reranker_model = AutoModelForSequenceClassification.from_pretrained(RERANKER_NAME)
reranker_model.to(device)
reranker_model.eval()


LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading LLM (4-bit)...")
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME, trust_remote_code=True)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

generator = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer
)

def retrieve_top_k(query: str, embedder, index, chunk_df, k: int = 20) -> pd.DataFrame:
    q_emb = embedder.encode(["query: " + query], normalize_embeddings=True)
    scores, indices = index.search(q_emb, k)
    scores, indices = scores[0], indices[0]
    valid = indices >= 0
    scores, indices = scores[valid], indices[valid]
    if len(indices) == 0:
        return pd.DataFrame()
    candidates = chunk_df.iloc[indices].copy()
    candidates['retrieval_score'] = scores
    return candidates

def rerank_chunks(query: str, chunks_df: pd.DataFrame,
                  top_k: int = 5, threshold: float = 0.3) -> List[Dict]:
    """Rerank candidates and return list of dicts with 'text' and 'score'."""
    if chunks_df.empty:
        return []
    pairs = [[query, text] for text in chunks_df['text']]
    with torch.no_grad():
        inputs = reranker_tokenizer(pairs, padding=True, truncation=True,
                                    return_tensors='pt').to(device)
        logits = reranker_model(**inputs, return_dict=True).logits.view(-1).float()
        scores = torch.sigmoid(logits).cpu().numpy()
    chunks_df['rerank_score'] = scores
    valid = chunks_df[chunks_df['rerank_score'] >= threshold]
    if valid.empty:
        return []
    best = valid.sort_values('rerank_score', ascending=False).head(top_k)
    return best[['text', 'rerank_score']].to_dict('records')


SYSTEM_PROMPT = (
    "Ты — ассистент банка, отвечающий на вопросы клиентов строго по предоставленным фрагментам документов.\n"
    "Правила:\n"
    "- Отвечай предельно кратко, без воды.\n"
    "- Используй markdown: * **ключ** значение, списки при необходимости.\n"
    "- Ссылайся на фрагменты: «Согласно Фрагменту 1…»\n"
    "- Никакой информации вне фрагментов.\n"
    "- Ответ не более 150 слов."
)

def build_prompt(query: str, fragments: List[Dict]) -> Optional[str]:
    """Create chat prompt with multiple fragments."""
    if not fragments:
        return None
    context_parts = []
    for i, frag in enumerate(fragments, start=1):
        context_parts.append(f"Фрагмент {i}:\n{frag['text']}")
    context = "\n\n".join(context_parts)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Вопрос: {query}\n\n{context}\n\nОтвет:"}
    ]
    prompt = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

def generate_batch(prompts: List[str], max_new_tokens: int = 180) -> List[str]:
    outputs = generator(
        prompts,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=llm_tokenizer.pad_token_id,
        return_full_text=False
    )
    return [o[0]['generated_text'] for o in outputs]



submission_qids = sample_sub['q_id'].tolist()
answers = []

BATCH_SIZE = 4
pending_prompts = []
results = {}

no_answer_count = 0

for qid in tqdm(submission_qids, desc="Processing"):
    query = qid2query.get(qid, "")
    if not query:
        results[qid] = "Нет ответа."
        continue

    candidates = retrieve_top_k(query, embedder, index, chunk_df, k=20)
    if candidates.empty:
        results[qid] = "Нет ответа."
        no_answer_count += 1
        continue

    frags = rerank_chunks(query, candidates, top_k=5, threshold=0.3)
    if not frags:
        results[qid] = "Нет ответа."
        no_answer_count += 1
        continue

    prompt = build_prompt(query, frags)
    pending_prompts.append((qid, prompt))

    if len(pending_prompts) >= BATCH_SIZE:
        prompts_batch = [p for _, p in pending_prompts]
        qids_batch = [q for q, _ in pending_prompts]
        generated = generate_batch(prompts_batch)
        import gc; gc.collect(); torch.cuda.empty_cache()
        for q, ans in zip(qids_batch, generated):
            results[q] = ans
        pending_prompts = []

if pending_prompts:
    prompts_batch = [p for _, p in pending_prompts]
    qids_batch = [q for q, _ in pending_prompts]
    generated = generate_batch(prompts_batch)
    import gc; gc.collect(); torch.cuda.empty_cache()
    for q, ans in zip(qids_batch, generated):
        results[q] = ans

for qid in submission_qids:
    if qid not in results:
        results[qid] = "Нет ответа."


submission_df = pd.DataFrame({
    'q_id': submission_qids,
    'answer_new': [results.get(qid, "Нет ответа.") for qid in submission_qids]
})

for col in sample_sub.columns:
    if col not in submission_df.columns:
        submission_df[col] = sample_sub[col]

submission_df = submission_df[sample_sub.columns]
print("Submission shape:", submission_df.shape)
print("Missing values:", submission_df.isnull().sum().sum())
print(f"No-answer responses: {no_answer_count} (filled automatically)")

OUTPUT_PATH = "/kaggle/working/submission.csv"
submission_df.to_csv(
    OUTPUT_PATH,
    index=False,
    quoting=csv.QUOTE_ALL,
    lineterminator='\n'
)
print(f"Submission saved to {OUTPUT_PATH}")

Device: cuda
GPU: Tesla T4
VRAM: 14.6 GB
Sample submission shape: (6977, 2)
Columns: ['q_id', 'answer_new']
Unique q_ids: 6977
✔ Each q_id appears exactly once.
Websites: (1937, 5)
Questions: (6977, 2)


Cleaning:   0%|          | 0/1937 [00:00<?, ?it/s]

Websites after cleaning: 1889


Chunking:   0%|          | 0/1889 [00:00<?, ?it/s]

Total chunks: 26430
Loading index and chunks from disk...


...

Loading LLM (4-bit)...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Processing:   0%|          | 0/6977 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set

Submission shape: (6977, 2)
Missing values: 0
No-answer responses: 4105 (filled automatically)
Submission saved to /content/submission.csv
